# COTS Component Selection & Freeze
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

Select the COTS hardware the conceptual design leaves open — **flight
controller, ESC, EDF drive motor, propeller (the fan/prop unit itself),
and the vane/aileron servo** — from the per-category database files in
`config/components/`, and **freeze** the winners with their mass and
own-CG inertia into `out/components.yaml`.

This is the freeze ADR-0011 anticipates: first flight is PX4 on a COTS
Pixhawk-class FC with a telemetry ESC, and "BEC current budgets and
connector choices … stop being placeholders once the COTS hardware list is
frozen". Once the parts below are procured, their ids get pinned via
`selection.frozen` in the config and the regression tests keep the design
honest against the real hardware.

| Input | Source |
|-------|--------|
| Converged design point (MTOW, \(P_{hover}\), \(P_{design}\)) | mass-closure re-run from `config/` |
| ESC current requirement | `compute_operating_point` (same law as NB10 / `out/electrical.yaml`) |
| Frozen rotor diameter | `config/rotor.yaml` (ADR-0003) |
| Servo torque requirements | `out/control_vanes.yaml`, `out/aileron.yaml` |
| Mass allocations | `out/fuselage.yaml` layout + `config/fuselage.yaml` |
| Candidate database + margins | `config/components/` (one YAML per category) |

| Output | Consumer |
|--------|----------|
| `out/components.yaml` (frozen parts, mass, \(\mathbf{I}_{CG}\)) | procurement, `px4/` bring-up, NB9 refinement, regression tests |
| `notebooks/figures/cots_budget_margins.png` | design report |

**Selection rule** (all physics in `src/conceptual_design/cots_selection.py`):
hard requirements are *derived* from the design point, never configured —
only the margins are config. Among candidates passing every requirement the
**lightest wins** (deterministic, ties by id). Mass-allocation fit is
**reported, never filtered on**: a too-heavy best candidate is a standing
finding against the weight fractions, exactly like the marginal ESC
cold-plate (ADR-0009).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

REPO_ROOT   = Path().resolve().parents[0]
SRC_PATH    = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config"
OUT_PATH    = REPO_ROOT / "out"
sys.path.insert(0, str(SRC_PATH))

plt.rcParams.update({
    "figure.dpi"        : 120,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.alpha"        : 0.25,
    "font.size"         : 10,
})
C = ["#2c7bb6", "#d7191c", "#1a9641", "#f68b33", "#762a83",
     "#5aae61", "#9970ab", "#e08214", "#35978f"]

FIG_DIR = Path("figures")   # per-notebook figures directory
FIG_DIR.mkdir(exist_ok=True)

# Section 1 — Design Inputs

Re-run the sizing loop from `config/` — same pattern as NB2–NB10, so this
notebook stays consistent with the upstream design state — then derive the
electrical operating point with the *same* function NB10 uses, and load the
servo-torque and packaging handoffs.

---

In [ ]:
from conceptual_design import (
    run_sizing_loop,
    Environment, Mission, Aerodynamics, Battery,
    WeightFraction, PropulsiveSystemParameters,
    ElectricalParams, compute_operating_point,
)
from conceptual_design.forward_flight_power import ForwardFlightParams
from conceptual_design.wing_sizing import WingStructureParams
from conceptual_design.models import RotorParams, Avionics

env     = Environment()
mission = Mission.from_yaml(CONFIG_PATH / "mission.yaml")
aero    = Aerodynamics.from_yaml(CONFIG_PATH / "aerodynamics.yaml")
batt    = Battery.from_yaml(CONFIG_PATH / "battery.yaml")
wf      = WeightFraction.from_yaml(CONFIG_PATH / "initial_weight_fraction_estimation.yaml")
prop    = PropulsiveSystemParameters.from_yaml(CONFIG_PATH / "propulsive_system_parameters.yaml")
ff      = ForwardFlightParams.from_yaml(CONFIG_PATH / "forward_flight_params.yaml")
ws      = WingStructureParams.from_yaml(CONFIG_PATH / "wing_structure_params.yaml")
rotor    = RotorParams.from_yaml(CONFIG_PATH / "rotor.yaml")
avionics = Avionics.from_yaml(CONFIG_PATH / "avionics.yaml")
elec     = ElectricalParams.from_yaml(CONFIG_PATH / "electrical.yaml")

result = run_sizing_loop(
    m_payload_kg = mission.payload_kg,
    mission      = mission,
    aero         = aero,
    batt         = batt,
    wf           = wf,
    prop_params  = prop,
    ff_params    = ff,
    ws_params    = ws,
    env          = env,
    D_rotor_m    = rotor.D_rotor_m,
    P_hotel_W    = avionics.P_hotel_W,
)
assert result.converged, "mass closure did not converge"

op = compute_operating_point(result, batt, elec)

# --- upstream handoffs ----------------------------------------------------
vanes   = yaml.safe_load((OUT_PATH / "control_vanes.yaml").read_text(encoding="utf-8"))
aileron = yaml.safe_load((OUT_PATH / "aileron.yaml").read_text(encoding="utf-8"))
fus     = yaml.safe_load((OUT_PATH / "fuselage.yaml").read_text(encoding="utf-8"))
fus_cfg = yaml.safe_load((CONFIG_PATH / "fuselage.yaml").read_text(encoding="utf-8"))
layout  = {e["name"]: e for e in fus["layout"]}

# NB10 writes the same operating point to out/electrical.yaml; if it is
# present it must agree (a mismatch means a stale handoff, not new physics).
elec_out_path = OUT_PATH / "electrical.yaml"
if elec_out_path.exists():
    elec_out = yaml.safe_load(elec_out_path.read_text(encoding="utf-8"))
    assert abs(elec_out["esc_rating_a"] - op.esc_rating_a) < 0.1, (
        "out/electrical.yaml is stale -- re-run the wiring_diagram notebook")

print(f"MTOW              : {result.m_total_kg:.3f} kg")
print(f"P_hover / P_design: {result.P_hover_W:.0f} / {result.P_design_W:.0f} W")
print(f"pack              : {elec.battery_series_cells}S  ({op.pack_voltage_v:.1f} V, "
      f"{op.pack_capacity_ah:.2f} Ah)")
print(f"hover current     : {op.hover_current_a:.1f} A -> ESC rating req {op.esc_rating_a:.1f} A")
print(f"servo torque req  : vane {vanes['servo_torque_req_gcm']:.0f} g cm, "
      f"aileron {aileron['servo_torque_req_gcm']:.0f} g cm")

# Section 2 — Derived Requirements

Every hard requirement comes from the converged design point; the database
contributes only the **margins** (`config/components/selection.yaml`):

| Category | Requirement | Basis |
|---|---|---|
| flight controller | PX4-supported, \(\geq\) 2 IMUs | ADR-0011 policy; IMU health matters in the ~211 Hz 1/rev environment (NB5) |
| ESC | \(I_{cont} \geq I_{hover} \cdot k_{esc}\), 6S in range, **live telemetry** | wiring-module current law; ADR-0011 flight-test campaign closes the loop on hover power via ESC telemetry |
| EDF motor | \(P_{max} \geq k_{mot} \cdot P_{design}\), 6S in range | hot-day derate + hover thrust-modulation headroom |
| propeller (fan/prop) | rotor diameter \(=\) `D_rotor_m`, \(P_{max} \geq k_{mot} \cdot P_{design}\) | ADR-0003 froze the diameter class — disk loading is *derived* from it, so a different diameter is a different design point, not an alternative part |
| servo | stall torque at the 5 V rail \(\geq k_{srv} \cdot \max(\tau_{vane}, \tau_{ail})\) | hinge moments from NB3/NB4; stall \(\neq\) usable torque |

kv ↔ rotor-map matching between motor and fan/prop needs the rotor curve
and is deliberately out of conceptual scope; NB1's empirical RPM law covers
it at this level.

---

In [ ]:
from conceptual_design.cots_selection import (
    CATEGORIES, ComponentDB, derive_requirements, select_components,
    budget_report, write_components_yaml,
)

db = ComponentDB.from_dir(CONFIG_PATH / "components")

reqs = derive_requirements(
    P_design_W          = result.P_design_W,
    esc_rating_a        = op.esc_rating_a,
    series_cells        = elec.battery_series_cells,
    D_rotor_m           = rotor.D_rotor_m,
    tau_vane_req_gcm    = vanes["servo_torque_req_gcm"],
    tau_aileron_req_gcm = aileron["servo_torque_req_gcm"],
    policy              = db.policy,
)

for cat, r in reqs.items():
    pretty = ", ".join(f"{k}={v:.1f}" if isinstance(v, float) else f"{k}={v}"
                       for k, v in r.items())
    print(f"{cat:18s}: {pretty}")

# Section 3 — Selection

All candidates per category, with the derived pass/fail verdict. The
lightest feasible part is selected (`config/components/selection.yaml` can
later pin a procured part via `selection.frozen`; the pin is still
re-validated on every run, so a design change that outgrows frozen hardware
fails loudly).

---

In [ ]:
selections = select_components(db, reqs)

RATING_COLS = {
    "flight_controller": ["imu_count", "px4"],
    "esc":               ["i_cont_a", "s_min", "s_max", "telemetry"],
    "edf_motor":         ["p_max_w", "s_min", "s_max", "kv"],
    "propeller":         ["d_rotor_mm", "p_max_w", "n_blades"],
    "servo":             ["stall_torque_gcm", "torque_at_v"],
}

for cat in CATEGORIES:
    sel = selections[cat]
    rows = []
    for spec in db.candidates[cat]:
        verdict = ("SELECTED" if spec.id == sel.selected.id
                   else "ok" if spec.id in sel.feasible_ids
                   else f"REJECTED: {sel.rejected[spec.id]}")
        row = {"id": spec.id, "mass_g": spec.mass_kg * 1e3}
        row.update({k: spec.ratings.get(k) for k in RATING_COLS[cat]})
        row["verdict"] = verdict
        rows.append(row)
    print(f"--- {cat}" + ("  [FROZEN by config]" if sel.frozen else ""))
    display(pd.DataFrame(rows).set_index("id"))

# Section 4 — Mass-Allocation Report

The frozen hardware against the weight-fraction allocations
(`out/fuselage.yaml` layout / `config/fuselage.yaml` carve-outs). These are
**findings, not filters** — the selection above already picked the lightest
part physics allows, so a negative margin here indicts the *allocation*,
not the part.

---

In [ ]:
n_servos = int(vanes["n_vanes"]) + int(aileron["n_ailerons"])

budgets = budget_report(
    selections, db,
    m_avionics_net_kg    = fus["m_avionics_net_kg"],
    m_esc_alloc_kg       = layout["esc"]["mass_kg"],
    m_motor_fan_alloc_kg = layout["motor_fan"]["mass_kg"],
    servo_alloc_kg       = fus_cfg["servo_mass_kg"],
    n_servos             = n_servos,
)

groups = ["avionics_bay", "esc", "motor_fan", "servo_each"]
labels = {
    "avionics_bay": "avionics bay\n(FC + GPS/RX/telemetry/...)",
    "esc":          "ESC",
    "motor_fan":    "EDF motor + fan/prop",
    "servo_each":   "servo (each, x%d)" % n_servos,
}

fig, ax = plt.subplots(figsize=(7.5, 3.6))
y = np.arange(len(groups))[::-1]
actual = [budgets[g]["actual_g"] for g in groups]
alloc  = [budgets[g]["alloc_g"]  for g in groups]
colors = [C[0] if budgets[g]["within"] else C[1] for g in groups]

ax.barh(y, actual, height=0.55, color=colors)
ax.plot(alloc, y, "k|", markersize=22, markeredgewidth=2.2, ls="none", zorder=3)
for yi, g in zip(y, groups):
    m = budgets[g]["margin_g"]
    ax.annotate(f"{m:+.0f} g", (max(budgets[g]['actual_g'], budgets[g]['alloc_g']), yi),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_yticks(y, [labels[g] for g in groups])
ax.set_xlabel("mass [g]")
ax.set_title("Frozen COTS hardware vs mass-closure allocations "
             "(bar = actual, tick = allocation; red = over)")
fig.tight_layout()
fig.savefig(FIG_DIR / "cots_budget_margins.png", bbox_inches="tight")
plt.show()

for g in groups:
    b = budgets[g]
    flag = "ok      " if b["within"] else "FINDING "
    print(f"{flag} {g:14s}: {b['actual_g']:6.1f} g vs {b['alloc_g']:6.1f} g "
          f"({b['margin_g']:+6.1f} g)  {b.get('note', '')}")

# Section 5 — Freeze: `out/components.yaml`

Each frozen part carries its inertia tensor about its **own CG** (solid box
/ solid cylinder of the catalog envelope, axes along the envelope's local
axes — mounting orientation is an installation detail for the NB9
mass-properties refinement). The regression tests pin the selected ids and
masses, so any silent change to the frozen hardware list turns CI red.

---

In [ ]:
rows = []
for cat, sel in selections.items():
    s = sel.selected
    I1, I2, I3 = s.inertia_cg()
    rows.append({
        "category": cat, "part": s.name, "mass_g": s.mass_kg * 1e3,
        "I1_kg_mm2": I1 * 1e6, "I2_kg_mm2": I2 * 1e6, "I3_kg_mm2": I3 * 1e6,
    })
display(pd.DataFrame(rows).set_index("category").round(3))

design_point = {
    "MTOW_kg":            round(result.m_total_kg, 4),
    "P_hover_W":          round(result.P_hover_W, 1),
    "P_design_W":         round(result.P_design_W, 1),
    "pack":               f"{elec.battery_series_cells}S",
    "D_rotor_mm":         round(rotor.D_rotor_m * 1e3, 1),
    "esc_rating_req_a":   round(op.esc_rating_a, 2),
    "servo_torque_req_gcm": round(max(vanes["servo_torque_req_gcm"],
                                      aileron["servo_torque_req_gcm"]), 2),
}
out_file = OUT_PATH / "components.yaml"
write_components_yaml(selections, budgets, db, design_point, out_file)

# --- self-checks on the written handoff ----------------------------------
frozen = yaml.safe_load(out_file.read_text(encoding="utf-8"))
assert set(frozen["selected"]) == set(CATEGORIES)
for cat, sel in selections.items():
    assert frozen["selected"][cat]["id"] == sel.selected.id
esc_sel = selections["esc"].selected
assert esc_sel.ratings["i_cont_a"] >= op.esc_rating_a
srv_sel = selections["servo"].selected
assert srv_sel.ratings["stall_torque_gcm"] >= reqs["servo"]["stall_torque_min_gcm"]
prop_sel = selections["propeller"].selected
assert abs(prop_sel.ratings["d_rotor_mm"] * 1e-3 - rotor.D_rotor_m) < 1.5e-3

print(f"wrote {out_file.relative_to(REPO_ROOT)}")
for cat, sel in selections.items():
    tag = "frozen" if sel.frozen else "auto"
    print(f"  {cat:18s} [{tag}] {sel.selected.id:26s} {sel.selected.mass_kg*1e3:6.1f} g")
print(f"all within allocations: {budgets['all_within']}")

# Findings & Conclusions

1. **Selected stack** (auto, lightest feasible at the current design
   point): Pixhawk 6C flight controller, APD 80F3[X] telemetry ESC,
   SunnySky X4120-class EDF motor, Schübeler DS-215-DIA HST fan (the only
   COTS rotor at the frozen 195 mm diameter), KST X08 vane/aileron
   servos. The ESC and servos fit their allocations comfortably; jet-vane
   torque — not aileron torque — sizes the servo.

2. **Standing finding — avionics bay over budget.** The lightest
   PX4-capable FC plus the ADR-0011 supporting stack (GPS, telemetry, RX,
   airspeed, power module, looms) exceeds the net avionics-bay budget by
   a few tens of grams. The avionics fraction was already flagged as
   near its floor when `fs_base` was re-baselined (ADR-0010); this puts a
   hardware-derived number on it.

3. **Dominant finding — the 195 mm COTS EDF class has no light
   implementation.** The only true COTS fan at the frozen diameter is the
   DS-215-DIA heavy-lift unit: fan-only it already outweighs the whole
   motor+fan allocation ~5×, and the motor sold with it is a 10 kW /
   3.4 kg part for 20 kg jets. Together with the lightest plausible
   motor, propulsion lands ~6× over its allocation. ADR-0003's "COTS
   195 mm EDF" is buyable only as heavy-lift hardware.

4. **The V-BAT-like resolution — a 3-blade prop in the airframe duct —
   is visible in the rejected table.** A COTS 3-blade 8×6 prop is ~23 g
   (~40× lighter than the fan cartridge) and the duct is airframe
   structure either way; at this vehicle's deliberately low disk loading
   the low solidity is appropriate. It is rejected **only on diameter**
   (COTS sizes bracket the frozen class: 178 / 203 mm vs 195 mm).
   Adopting it is a *design change, not a part swap*: move `D_rotor_m`
   to 0.203 (or 0.178), re-converge the pipeline, revisit
   solidity/figure-of-merit and the empirical RPM law, and amend
   ADR-0003 — deliberately not smuggled in through a parts database.
   This is the recommended path; the propulsion fraction only closes
   with a prop-in-duct build.

5. **Freeze procedure.** At procurement: weigh the real parts, correct
   `config/components/` (every EST entry), set `selection.frozen` ids,
   and update the regression pins in the same commit — the diff is the
   procurement record.